# Hybrid RAG

Combining the [langchain-rag](https://python.langchain.com/docs/tutorials/rag/) with the 
[hybrid rag](https://pub.towardsai.net/hybrid-rag-made-easy-step-by-step-with-langchain-faiss-azureopenai-llmgraphtransformer-and-ef93cd50948d)

In [1]:
import os
from langchain_experimental.graph_transformers import LLMGraphTransformer
import networkx as nx
from langchain.chains import GraphQAChain
from langchain_core.documents import Document
from langchain_community.graphs.networkx_graph import NetworkxEntityGraph
from langchain.chains import RetrievalQA
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_core.vectorstores import InMemoryVectorStore

from langchain.vectorstores import FAISS
from langchain.document_loaders import TextLoader
from langchain_ollama import ChatOllama
from langchain_ollama import OllamaEmbeddings
import json
from langgraph.graph import START, StateGraph
from typing_extensions import List, TypedDict
import pandas as pd
from langchain_core.documents import Document
from langchain_ollama import ChatOllama
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings
from langchain.prompts import ChatPromptTemplate
from langchain_core.prompts import PromptTemplate
import chroma



## Set up the basic RAG

### create the vector store

In [2]:

llm_model = "llama3.2:latest"
llm = ChatOllama(
   model=llm_model,
   temperature=0,
   # other params...
)


def build_vector_store(documents,filename,embeddings):

    vector_store = Chroma(
    collection_name="example_collection",
        embedding_function=embeddings,
     persist_directory="./"+filename,  # Where to save data locally, remove if not necessary
     )
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    all_splits = text_splitter.split_documents(training_documents)
    _ = vector_store.add_documents(documents=all_splits)
    return vector_store

print("starting to load")
with open("./data/training_modeling_papers.json", "r") as f:
    data = json.load(f)

print("read json..."+str(len(data)))
training_documents = []

for row in data:
    training_documents.append(Document(page_content=row["abstract"]))

print("documents created..."+str(len(training_documents)))
f"Papers loaded: {len(training_documents)}"



starting to load
read json...46
documents created...46


'Papers loaded: 46'

In [3]:
print("starting to load")
with open("./data/modeling_papers.json", "r") as f:
    data = json.load(f)

print("read json..."+str(len(data)))
modeling_documents = []

for row in data:
    modeling_documents.append(Document(page_content=row["abstract"]))

print("documents created..."+str(len(modeling_documents)))
f"Papers loaded: {len(modeling_documents)}"

starting to load
read json...5737
documents created...5737


'Papers loaded: 5737'

In [4]:
train2 = modeling_documents[:200]

In [5]:
training_documents[0]

Document(metadata={}, page_content='Background: Since the appearance of the first case of COVID-19 in Morocco, the cumulative number of reported infectious cases continues to increase and, consequently, the government imposed the containment measure within the country. Our aim is to predict the impact of the compulsory containment on COVID-19 spread. Earlier knowledge of the epidemic characteristics of COVID-19 transmission related to Morocco will be of great interest to establish an optimal plan-of-action to control the epidemic.\n\nMethod: Using a Susceptible-Asymptomatic-Infectious model and the data of reported cumulative confirmed cases in Morocco from March 2nd to April 9, 2020, we determined the basic and control reproduction numbers and we estimated the model parameter values. Furthermore, simulations of different scenarios of containment are performed.\n\nResults: Epidemic characteristics are predicted according to different rates of containment. The basic reproduction number 

In [6]:
embeddings = OllamaEmbeddings(model=llm_model)

vector_store = build_vector_store(train2,"chroma_langchain2.db",embeddings)


### build the RAG retrieval

In [7]:

def create_generic_rag(vector_store):
    template = """Use the following pieces of context to summarize the question provided at the end.

    {context}

    Question: {question}

    Helpful Answer:"""

    custom_rag_prompt = PromptTemplate.from_template(template)

    # set up state
    # Define state for application
    class State(TypedDict):
        question: str
        context: List[Document]
        answer: str


    # Define application steps
    def retrieve(state: State):
        retrieved_docs = vector_store.similarity_search(state["question"])
        return {"context": retrieved_docs}


    def generate(state: State):
        docs_content = "\n\n".join(doc.page_content for doc in state["context"])
        messages = custom_rag_prompt.invoke({"question": state["question"], "context": docs_content})
        response = llm.invoke(messages)
        return {"answer": response.content}

    # Compile application and test
    graph_builder = StateGraph(State).add_sequence([retrieve, generate])
    graph_builder.add_edge(START, "retrieve")
    generic_rag = graph_builder.compile()
    return generic_rag

In [8]:
generic_rag = create_generic_rag(vector_store)
generic_rag.invoke({"question": training_documents[0].page_content})

{'question': 'Background: Since the appearance of the first case of COVID-19 in Morocco, the cumulative number of reported infectious cases continues to increase and, consequently, the government imposed the containment measure within the country. Our aim is to predict the impact of the compulsory containment on COVID-19 spread. Earlier knowledge of the epidemic characteristics of COVID-19 transmission related to Morocco will be of great interest to establish an optimal plan-of-action to control the epidemic.\n\nMethod: Using a Susceptible-Asymptomatic-Infectious model and the data of reported cumulative confirmed cases in Morocco from March 2nd to April 9, 2020, we determined the basic and control reproduction numbers and we estimated the model parameter values. Furthermore, simulations of different scenarios of containment are performed.\n\nResults: Epidemic characteristics are predicted according to different rates of containment. The basic reproduction number is estimated to be 2.9

## build the graphrag

In [9]:
def build_graph_rag(llm,training_documents):

    llm_transformer = LLMGraphTransformer(llm=llm)
    graph_documents = llm_transformer.convert_to_graph_documents(training_documents)

    graph = NetworkxEntityGraph()

    for node in graph_documents[0].nodes:
        graph.add_node(node.id)

    for edge in graph_documents[0].relationships:
        graph._graph.add_edge(
            edge.source.id,
            edge.target.id,
            relation=edge.type
        )

        graph._graph.add_edge(
            edge.target.id,
            edge.source.id,
            relation=edge.type+" by",
        )
    graph_rag = GraphQAChain.from_llm(
            llm=llm,
            graph=graph,
            verbose=True
        )
    return graph_rag

In [10]:
graph_rag = build_graph_rag(llm,training_documents)

KeyboardInterrupt: 

In [ ]:
def hybrid_rag(generic_rag,graph_rag):
    def rag_processor(query):
        # Doing generic RAG
        rag_result = generic_rag.invoke({"question": query})
    
        graph_rag_result = graph_rag.invoke(query)
        prompt = """You are a helpful assistant.
        Generate a summary of the passage provided, using the two contexts provided:

        Context 1: {}
        Context 2: {} 

        Question: {}
    
        """.format(rag_result,graph_rag_result,query)

        return llm.invoke(prompt),rag_result,graph_rag_result
    return rag_processor

In [ ]:
hr = hybrid_rag(generic_rag,graph_rag)

In [ ]:
def hybrid_rag2(graph_rag):
    def rag_processor(query):
        # Doing generic RAG
        graph_rag_result = graph_rag.invoke(query)
        prompt = """You are a helpful assistant.
        Generate a summary of the passage provided, using the two contexts provided:

        Context 1: {}
       
        Question: {}
    
        """.format(rag_result,graph_rag_result,query)

        return llm.invoke(prompt),rag_result,graph_rag_result
    return rag_processor

In [ ]:
hr2 =hybrid_rag2(graph_rag)

# routines for printing results and getting papers

In [ ]:
def print_graph_results(graph_documents: list[Document]) -> None:
    for doc in graph_documents:
        if len(doc.nodes) > 0:
            print(f"Paper ID: {doc.source.id}")
            print(f"Paper Abstract: {doc.source.page_content}")

            for node in doc.nodes:
                print(node)
                print(f"Node: {node.id}, Type: {node.type}")

            for rel in doc.relationships:
                print(f"Relationship: {rel.type}")
                print(f"   Source: {rel.source.id}, Type: {rel.source.type}")
                print(f"   Target: {rel.target.id}, Type: {rel.target.type}")

            print()


def pgr(graph_documents: list[Document]) -> None:
    for doc in graph_documents:
        if len(doc.nodes) > 0 and len(doc.relationships) > 0:
            for rel in doc.relationships:
                print(f"   Source: {rel.source.id} ({rel.source.type}) -> {rel.type} -> {rel.target.id} ({rel.target.type})")
        print()

In [ ]:
import pandas as pd

df_modeling_papers = pd.read_json("./data/modeling_papers_0.json", orient="records", lines=True)

documents = []

for row in df_modeling_papers.itertuples():
    documents.append(Document(id=row.id, page_content=row.abstract))

f"Papers loaded: {len(documents)}"

In [ ]:
documents[0]

In [ ]:
res,rag_res,graph_res = hr(documents[0].page_content)

In [ ]:
res.content

## run extracted summary through graph extraction

In [ ]:
example_out = [Document(page_content=res.content)]
transformer = LLMGraphTransformer(llm=llm)
graph_documents = transformer.convert_to_graph_documents(example_out)

In [ ]:
pgr(graph_documents)

In [ ]:
direct_graph_extraction=transformer.convert_to_graph_documents([documents[0]])
pgr(direct_graph_extraction)

In [ ]:
documents[0]